# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [3]:
import pandas as pd

df = pd.read_csv('/Users/anshusharma1705/Desktop/machine-Learning/Lab/mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [4]:
print(df.info())
print("shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB
None
shape: (398, 9)

Missing values:
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


How many cars from each `origin`? Use `value_counts()`.

In [5]:
print(df['origin'].value_counts())

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64


---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [6]:
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
df = df.drop(columns=['name'])
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa


The `name` column is an identifier/text description of the car, not a useful numeric specification. Keeping it would not give KNN a meaningful numeric feature and could lead to overfitting.

---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [7]:
X = df.drop(columns=['origin'])
y = df['origin']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", list(X.columns))
print("Target:", y.name)

X shape: (398, 7)
y shape: (398,)
Features: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
Target: origin


---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (318, 7)
X_test: (80, 7)
y_train: (318,)
y_test: (80,)


---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [9]:
print(X_train.min())
print()
print(X_train.max())
print("\nRange of each feature:")
print(X_train.max() - X_train.min())

ranges = X_train.max() - X_train.min()
print("\nLargest range:", ranges.idxmax(), "=", ranges.max())

mpg                9.0
cylinders          3.0
displacement      68.0
horsepower        46.0
weight          1613.0
acceleration       8.0
model_year        70.0
dtype: float64

mpg               46.6
cylinders          8.0
displacement     455.0
horsepower       225.0
weight          5140.0
acceleration      24.8
model_year        82.0
dtype: float64

Range of each feature:
mpg               37.6
cylinders          5.0
displacement     387.0
horsepower       179.0
weight          3527.0
acceleration      16.8
model_year        12.0
dtype: float64

Largest range: weight = 3527.0


Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled test data shape:", X_test_scaled.shape)

Scaled training data shape: (318, 7)
Scaled test data shape: (80, 7)


---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_unscaled = KNeighborsClassifier()
knn_unscaled.fit(X_train, y_train)

y_pred_unscaled = knn_unscaled.predict(X_test)
accuracy_unscaled = accuracy_score(y_test, y_pred_unscaled)

print("Unscaled accuracy:", accuracy_unscaled)

Unscaled accuracy: 0.6875


**Now with scaling.** Same model, same `k`, scaled data.

In [12]:
knn_scaled = KNeighborsClassifier()
knn_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = knn_scaled.predict(X_test_scaled)
accuracy_scaled = accuracy_score(y_test, y_pred_scaled)

print("Scaled accuracy:", accuracy_scaled)

Scaled accuracy: 0.75


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

In [13]:
print("Unscaled accuracy:", accuracy_unscaled)
print("Scaled accuracy:  ", accuracy_scaled)
print("Difference:       ", accuracy_scaled - accuracy_unscaled)

if accuracy_scaled > accuracy_unscaled:
    print("Higher accuracy: scaled")
elif accuracy_unscaled > accuracy_scaled:
    print("Higher accuracy: unscaled")
else:
    print("Both accuracies are equal.")

Unscaled accuracy: 0.6875
Scaled accuracy:   0.75
Difference:        0.0625
Higher accuracy: scaled


Scaling changed the distances used by KNN. Because the features have very different ranges, scaling prevents large-range features from dominating the distance calculation.

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [14]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
k_results = {}

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    accuracy = knn.score(X_test_scaled, y_test)
    k_results[k] = accuracy
    print(f"k={k}: accuracy={accuracy:.4f}")

best_k = max(k_results, key=k_results.get)
best_accuracy = k_results[best_k]

print(f"\nBest k: {best_k}")
print(f"Best accuracy: {best_accuracy:.4f}")

k=1: accuracy=0.7375
k=3: accuracy=0.7500
k=5: accuracy=0.7500
k=7: accuracy=0.7500
k=9: accuracy=0.7500
k=11: accuracy=0.7250
k=15: accuracy=0.7375
k=21: accuracy=0.7250

Best k: 3
Best accuracy: 0.7500


**Best k:** Run the cell above; it prints the best `k` and its accuracy.

The accuracy can be compared across the printed values to decide whether it changes a lot or stays roughly flat.

Use the output from the previous cell:

- **Best k:** the value printed as `Best k`.
- **Its accuracy:** the value printed as `Best accuracy`.
- **Across k:** compare the eight printed accuracy values. If they are close, accuracy stays roughly flat; if they vary substantially, it changes a lot.